# 01｜Capital Bikeshare 資料健檢
2025 年 1 月官方旅次檔。檢查以旗標計數，保留原始資料。

In [1]:
from pathlib import Path
from zipfile import ZipFile
from hashlib import sha256
import pandas as pd
from IPython.display import display

zip_path = next(p for p in (Path('../data/raw'), Path('data/raw')) if p.is_dir()) / '202501-capitalbikeshare-tripdata.zip'
with ZipFile(zip_path) as archive:
    members = [p for p in archive.namelist() if p.lower().endswith('.csv') and not p.startswith('__MACOSX/')]
    assert len(members) == 1, members
    with archive.open(members[0]) as source:
        rides = pd.read_csv(source, dtype='string', keep_default_na=False)
print(members[0], rides.shape)
print('SHA-256:', sha256(zip_path.read_bytes()).hexdigest())
display(pd.DataFrame({'dtype': rides.dtypes.astype(str), 'missing_or_blank': rides.apply(lambda s: s.str.strip().eq('').sum())}))

202501-capitalbikeshare-tripdata.csv (285731, 13)
SHA-256: d1bb7539d0a2b016bfdce2bc3db60b801869948ecf5139a32f897422ee6d312e


,dtype,missing_or_blank
ride_id,string,0
rideable_type,string,0
started_at,string,0
ended_at,string,0
start_station_name,string,88122
start_station_id,string,88122
end_station_name,string,91022
end_station_id,string,91217
start_lat,string,0
start_lng,string,0


ID 以字串讀取，保留前導零及原始格式。空白含純空白。時間以原始檔案的無時區時間解析，尚未套用時區或補值。

In [2]:
def blank(column):
    return rides[column].str.strip().eq('')

ride_id = rides.ride_id.str.strip().mask(blank('ride_id'))
started = pd.to_datetime(rides.started_at.mask(blank('started_at')), format='mixed', errors='coerce')
ended = pd.to_datetime(rides.ended_at.mask(blank('ended_at')), format='mixed', errors='coerce')
minutes = (ended - started).dt.total_seconds() / 60
start_missing = blank('start_station_id')
end_missing = blank('end_station_id')
checks = {
    'missing_ride_id': ride_id.isna().sum(),
    'duplicate_ride_id_rows_all_copies': (ride_id.notna() & ride_id.duplicated(keep=False)).sum(),
    'duplicate_ride_id_distinct_keys': ride_id[ride_id.duplicated(keep=False)].nunique(),
    'started_parse_failed_nonblank': (started.isna() & ~blank('started_at')).sum(),
    'ended_parse_failed_nonblank': (ended.isna() & ~blank('ended_at')).sum(),
    'duration_unavailable': minutes.isna().sum(),
    'duration_negative': minutes.lt(0).sum(),
    'duration_zero': minutes.eq(0).sum(),
    'duration_positive_under_1_min': (minutes.gt(0) & minutes.lt(1)).sum(),
    'duration_over_24h': minutes.gt(1440).sum(),
    'missing_start_station_id': start_missing.sum(),
    'missing_end_station_id': end_missing.sum(),
    'missing_both_station_ids': (start_missing & end_missing).sum(),
    'missing_either_station_id': (start_missing | end_missing).sum(),
    'both_station_ids_present': (~start_missing & ~end_missing).sum(),
    'start_id_distinct': rides.start_station_id.str.strip()[~start_missing].nunique(),
    'end_id_distinct': rides.end_station_id.str.strip()[~end_missing].nunique(),
    'start_name_present_id_missing': (~blank('start_station_name') & start_missing).sum(),
    'end_name_present_id_missing': (~blank('end_station_name') & end_missing).sum(),
}
for key, value in checks.items():
    print(f'{key}: {value:,}')
print('Parsed duration summary (minutes):')
print(minutes.describe(percentiles=[.01,.5,.95,.99]).round(2).to_string())
print('Time coverage:', started.min(), started.max(), ended.min(), ended.max())

missing_ride_id: 0
duplicate_ride_id_rows_all_copies: 0
duplicate_ride_id_distinct_keys: 0
started_parse_failed_nonblank: 0
ended_parse_failed_nonblank: 0
duration_unavailable: 0
duration_negative: 0
duration_zero: 0
duration_positive_under_1_min: 5,899
duration_over_24h: 190
missing_start_station_id: 88,122
missing_end_station_id: 91,217
missing_both_station_ids: 60,381
missing_either_station_id: 118,958
both_station_ids_present: 166,773
start_id_distinct: 773
end_id_distinct: 770
start_name_present_id_missing: 0
end_name_present_id_missing: 195
Parsed duration summary (minutes):
count    285731.00
mean         12.19
std          45.62
min           0.00
1%            0.35
50%           7.88
95%          28.52
99%          55.05
max        1499.96
Time coverage: 2024-12-31 19:50:05.679000 2025-01-31 23:59:45.752000 2025-01-01 00:00:37.974000 2025-01-31 23:59:59.223000


24 小時與正值小於 1 分鐘是人工複查門檻。欄位完整性與 ID 唯一性不等於站點身分已與官方站表核對。各旗標可能重疊。下一階段才建立借還事件及站點小時資料。